## Historical Context — One-Time Historical Data Pull (Do Not Re-Run)

### What this notebook did

Before this project was built, a short test scrape had produced daily price data 
for 1,101 tokens covering only a narrow recent window: **14 January – 11 February 2026**.

This notebook called the CoinMarketCap OHLCV Historical API to request the 
missing 24 months of price history for those same 1,101 tokens, going back to 
**February 2024**. The API allows historical queries — you specify a token ID and 
a date range, and it returns past prices for those dates. The data was requested 
in four chunks (each covering ~6 months) to stay within API limits:

- February 2024 → August 2024  
- August 2024 → February 2025  
- February 2025 → August 2025  
- August 2025 → January 2026  

These four chunks were then joined with the existing one-month file to produce 
a single continuous dataset spanning **February 2024 to February 2026**.

The two input files it required were produced by that earlier test scrape and 
are stored on a separate drive not included in this repository:

- `cmc_ohlcv_daily.csv` — the short one-month OHLCV file (the starting point)
- `cmc_token_level_panel.csv` — used only to extract the list of 1,101 token IDs

### Output (already saved)

The combined result is saved at:

    raw/cmc_ohlcv_daily_extended_24m.csv

784,004 rows · 1,101 tokens · February 2024 – February 2026.  
This file is read by NB04 and does not need to be regenerated.

### Why this notebook cannot be re-run

The two input files above are not present in this repository. Re-running today 
would also produce a different date range, since the 24-month lookback window 
is calculated relative to the current date. The saved output in `raw/` is the 
correct and fixed input for all downstream notebooks.

In [ ]:
import os
import time
from pathlib import Path
from datetime import datetime, timedelta, date

import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv()

# ======================
# CONFIG
# ======================
FINAL_PANEL_PATH = Path("cmc_token_level_panel.csv")
EXISTING_OHLCV   = Path("cmc_ohlcv_daily.csv")
OUTDIR = Path("outputs_cmc"); OUTDIR.mkdir(exist_ok=True)
OUT_EXTENDED_OHLCV = OUTDIR / "cmc_ohlcv_daily_extended_24m.csv"

CMC_API_KEY = os.getenv("CMC_API_KEY")
if not CMC_API_KEY:
    raise RuntimeError("Set CMC_API_KEY in env (.env is fine).")

BASE_URL = "https://pro-api.coinmarketcap.com"
HEADERS = {"Accept": "application/json", "X-CMC_PRO_API_KEY": CMC_API_KEY}
OHLCV_HIST = f"{BASE_URL}/v1/cryptocurrency/ohlcv/historical"

# ---- Plan constraint: 24 months max lookback ----
PLAN_LOOKBACK_DAYS = 730

# Request tuning
ID_CHUNK_SIZE = 50
WINDOW_DAYS   = 180
MAX_RETRIES   = 6
BASE_SLEEP    = 1.0

# ======================
# HELPERS
# ======================
def chunk_list(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

def cmc_get(params, retries=MAX_RETRIES):
    for k in range(retries):
        r = requests.get(OHLCV_HIST, headers=HEADERS, params=params, timeout=60)
        if r.status_code == 200:
            return r.json()
        if r.status_code in (429, 500, 502, 503, 504):
            time.sleep(BASE_SLEEP * (2 ** k))
            continue
        raise RuntimeError(f"CMC {r.status_code}: {r.text}")
    raise RuntimeError("CMC failed after retries.")

def parse_ohlcv(payload):
    data = payload.get("data", {})
    if isinstance(data, dict) and "quotes" in data:
        data = {str(data.get("id")): data}

    rows = []
    if not isinstance(data, dict):
        return pd.DataFrame()

    for _, obj in data.items():
        cid = obj.get("id")
        if cid is None:
            continue
        cid = int(cid)

        for q in obj.get("quotes") or []:
            usd = (q.get("quote", {}) or {}).get("USD", {}) or {}
            ts = usd.get("timestamp") or q.get("time_close") or q.get("time_open")
            if not ts:
                continue
            d = pd.to_datetime(ts, utc=True, errors="coerce")
            if pd.isna(d):
                continue

            rows.append({
                "cmc_id": cid,
                "date": d.date(),
                "open": usd.get("open"),
                "high": usd.get("high"),
                "low": usd.get("low"),
                "close": usd.get("close"),
                "volume": usd.get("volume"),
                "market_cap": usd.get("market_cap"),
            })

    out = pd.DataFrame(rows)
    if out.empty:
        return out

    for c in ["open", "high", "low", "close", "volume", "market_cap"]:
        out[c] = pd.to_numeric(out[c], errors="coerce")
    return out

# ======================
# MAIN
# ======================
# 1) final token set
final_panel = pd.read_csv(FINAL_PANEL_PATH)
final_ids = (
    pd.to_numeric(final_panel["cmc_id"], errors="coerce")
      .dropna().astype(int).unique().tolist()
)
final_ids = sorted(final_ids)
print("Final token set size:", len(final_ids))

# 2) existing OHLCV
if EXISTING_OHLCV.exists():
    existing = pd.read_csv(EXISTING_OHLCV)
    existing["date"] = pd.to_datetime(existing["date"], errors="coerce").dt.date
    existing = existing.dropna(subset=["cmc_id","date"]).drop_duplicates(["cmc_id","date"])
    existing = existing[existing["cmc_id"].isin(final_ids)].copy()
else:
    existing = pd.DataFrame(columns=["cmc_id","date","open","high","low","close","volume","market_cap"])

if not existing.empty:
    current_min_date = existing["date"].min()
    current_max_date = existing["date"].max()
    print("Existing OHLCV:", current_min_date, "->", current_max_date)
else:
    current_min_date = datetime.utcnow().date()
    current_max_date = datetime.utcnow().date()
    print("No existing OHLCV found; will pull only within plan window.")

# 3) enforce plan start boundary
today = datetime.utcnow().date()
plan_min_date = today - timedelta(days=PLAN_LOOKBACK_DAYS)

# we backfill strictly before what we already have, but not before plan_min_date
target_end = current_min_date
target_start = max(plan_min_date, target_end - timedelta(days=PLAN_LOOKBACK_DAYS))

print("Plan minimum date allowed:", plan_min_date)
print("Backfill range:", target_start, "->", target_end)

if target_start >= target_end:
    print("Nothing to backfill (already at plan limit or no earlier data needed).")
    extended = existing.sort_values(["cmc_id","date"])
    extended.to_csv(OUT_EXTENDED_OHLCV, index=False)
    print("Saved:", OUT_EXTENDED_OHLCV)
    raise SystemExit

# 4) build windows forward
windows = []
cursor = target_start
while cursor < target_end:
    w_start = cursor
    w_end = min(cursor + timedelta(days=WINDOW_DAYS), target_end)
    windows.append((w_start, w_end))
    cursor = w_end

print("Windows:", len(windows), "of", WINDOW_DAYS, "days each")

# 5) fetch
all_parts = []
for wi, (w_start, w_end) in enumerate(windows, start=1):
    time_start = (w_start - timedelta(days=1)).isoformat()  # exclusive
    time_end   = w_end.isoformat()                          # inclusive
    print(f"\nWindow {wi}/{len(windows)}: {w_start} -> {w_end}")

    for ids_chunk in chunk_list(final_ids, ID_CHUNK_SIZE):
        params = {
            "id": ",".join(map(str, ids_chunk)),
            "time_period": "daily",
            "time_start": time_start,
            "time_end": time_end,
            "convert": "USD",
            "skip_invalid": "true",
        }
        payload = cmc_get(params)
        part = parse_ohlcv(payload)
        if not part.empty:
            all_parts.append(part)
        time.sleep(0.25)

backfill = pd.concat(all_parts, ignore_index=True) if all_parts else pd.DataFrame()
if not backfill.empty:
    backfill = backfill.dropna(subset=["cmc_id","date"]).drop_duplicates(["cmc_id","date"])
    print("\nBackfill rows:", backfill.shape[0])
else:
    print("\nNo backfill rows returned.")

extended = pd.concat([existing, backfill], ignore_index=True)
extended = (
    extended.dropna(subset=["cmc_id","date"])
            .drop_duplicates(["cmc_id","date"])
            .sort_values(["cmc_id","date"])
)

extended.to_csv(OUT_EXTENDED_OHLCV, index=False)
print("\nSaved extended OHLCV:", OUT_EXTENDED_OHLCV)
print("Extended shape:", extended.shape)
print("Extended date range:", extended["date"].min(), "->", extended["date"].max())
print("Unique tokens:", extended["cmc_id"].nunique(), "/", len(final_ids))

C:\Users\paolina.gocheva\AppData\Local\Temp\ipykernel_14080\2673008606.py:102: DtypeWarning: Columns (23,26,27,41,42,43,45) have mixed types. Specify dtype option on import or set low_memory=False.
  final_panel = pd.read_csv(FINAL_PANEL_PATH)
C:\Users\paolina.gocheva\AppData\Local\Temp\ipykernel_14080\2673008606.py:129: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  today = datetime.utcnow().date()


Final token set size: 1101
Existing OHLCV: 2026-01-14 -> 2026-02-11
Plan minimum date allowed: 2024-02-13
Backfill range: 2024-02-13 -> 2026-01-14
Windows: 4 of 180 days each

Window 1/4: 2024-02-13 -> 2024-08-11

Window 2/4: 2024-08-11 -> 2025-02-07

Window 3/4: 2025-02-07 -> 2025-08-06

Window 4/4: 2025-08-06 -> 2026-01-14

Backfill rows: 753708

Saved extended OHLCV: outputs_cmc\cmc_ohlcv_daily_extended_24m.csv
Extended shape: (784004, 8)
Extended date range: 2024-02-13 -> 2026-02-11
Unique tokens: 1101 / 1101
